# 03 — Logistic Regression · Overload Prediction

**Input:** `hive_metastore.gold.gold_features` (pre-scaled, with `split` column)  
**Tracking:** MLflow experiment `Transformer_Overload`

### Notebook structure (shared across all models)

| Section | Description |
|---|---|
| 1 | Configuration & imports |
| 2 | Load data (train/test from `split` column) |
| 3 | Feature assembly pipeline |
| 4 | Class imbalance handling |
| 5 | Hyperparameter grid search (with MLflow) |
| 6 | Best model evaluation on test set |
| 7 | Threshold sweep |
| 8 | Confusion matrix & curves |
| 9 | Feature importance (individual + grouped) |
| 10 | Summary |

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/03_gold_features/00_Evaluation

## 1 · Configuration & imports

In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — shared constants across all model notebooks
# ══════════════════════════════════════════════════════════════════════════════
SRC_TABLE    = "hive_metastore.gold.gold_features"
ID_COL       = "ID_prefix"
TS_COL       = "DATE"
LABEL_COL    = "label_4h"            # change to "label_24h" for 24 h horizon
MODEL_NAME   = "LogisticRegression"
EXPERIMENT = "/Users/daniel.branco@cgi.com/Transformer_Overload_val"
SEED         = 42

# ── Feature groups (must match 02_gold_features) ─────────────────────────────
SIGNAL_COLS = ["current", "voltage"]

LOAD_RATIO_COLS = ["load_ratio_c", "load_ratio_v"]

ROLLING_COLS = [
    f"{s}_{stat}_{w}"
    for s in SIGNAL_COLS
    for stat in ["mean", "std", "max"]
    for w in ["1h", "1d", "7d"]
]

LAG_COLS = [
    f"{s}_lag_{l}"
    for s in SIGNAL_COLS
    for l in ["15m", "1h", "1d"]
]

WEATHER_RAW_COLS = [
    "temperatura_media_do_ar_horaria_c",
    "precipitacao_horaria_mm",
    "humidade_relativa_media_horaria_percent",
    "velocidade_do_vento_media_horaria_m_per_s",
]

WEATHER_DERIVED_COLS = ["temp_mean_1d", "temp_mean_7d", "precip_sum_1d"]

TEMPORAL_COLS = ["hour", "day_of_week", "month", "is_weekend"]

EVENT_COLS = ["events_15m_cnt"]

# All numeric features fed to the model (already scaled in gold_features)
NUMERIC_FEATURE_COLS = (
    LOAD_RATIO_COLS
    + ROLLING_COLS
    + LAG_COLS
    + WEATHER_RAW_COLS
    + WEATHER_DERIVED_COLS
    + TEMPORAL_COLS
    + EVENT_COLS
)

CAT_COLS = [ID_COL, "CONCELHO"]

N_HASH_BUCKETS = 256

# ── Sets used by classify_feature() in 00_Evaluation ─────────────────────────
WEATHER_RAW_SET     = set(WEATHER_RAW_COLS)
WEATHER_DERIVED_SET = set(WEATHER_DERIVED_COLS)
LOAD_RATIO_SET      = set(LOAD_RATIO_COLS)
TEMPORAL_SET        = set(TEMPORAL_COLS)
EVENT_SET           = set(EVENT_COLS)

print(f"Numeric features : {len(NUMERIC_FEATURE_COLS)}")
print(f"Categorical cols : {CAT_COLS}")
print(f"Label            : {LABEL_COL}")

## 2 · Load data (train/test from `split` column)

The chronological split, purge gap, and z-score scaling are all done in  
`02_gold_features`. We just filter by the `split` column here.

In [0]:
df = spark.read.table(SRC_TABLE)
print(f"Loaded {df.count():,} rows  |  {len(df.columns)} columns")

# Keep only needed columns + drop any residual nulls
keep_cols = NUMERIC_FEATURE_COLS + [LABEL_COL] + CAT_COLS + [TS_COL, "split"]

# Filter to only columns that exist (safety check)
existing = set(df.columns)
missing = set(NUMERIC_FEATURE_COLS) - existing
if missing:
    print(f"⚠️  Missing columns (removed from feature list): {missing}")
    NUMERIC_FEATURE_COLS[:] = [c for c in NUMERIC_FEATURE_COLS if c in existing]

keep_cols = [c for c in keep_cols if c in existing]
df_clean = df.select(keep_cols).dropna()
print(f"After dropna: {df_clean.count():,} rows")

In [0]:
train_df = df_clean.filter(F.col("split") == "train")
test_df  = df_clean.filter(F.col("split") == "test")

print(f"Train : {train_df.count():,} rows")
print(f"Test  : {test_df.count():,} rows")

for name, sdf in [("Train", train_df), ("Test", test_df)]:
    total = sdf.count()
    pos   = sdf.filter(F.col(LABEL_COL) == 1).count()
    neg   = total - pos
    print(f"\n{name}: {pos:,} pos ({100*pos/total:.2f}%)  |  {neg:,} neg ({100*neg/total:.2f}%)")

## 3 · Feature assembly pipeline

Since all numeric features are **already z-scored** in `02_gold_features`,  
the pipeline only needs to:

1. StringIndex + hashing the categorical columns (`ID_prefix`, `CONCELHO`)  
2. Assemble numerics + hashing into a single `features` vector  
3. Feed into Logistic Regression (`standardization=False`)

In [0]:
def make_lr_pipeline(reg_param=0.0, elastic_net=0.0, max_iter=200):
    """
    Build a PySpark ML pipeline for Logistic Regression.
    Uses FeatureHasher for categoricals (memory-safe with 1000+ IDs).
    No StandardScaler — numerics arrive pre-scaled from gold_features.
    """
    # Categorical → fixed-size hashed vector
    hasher = FeatureHasher(
        inputCols=CAT_COLS,
        outputCol="cat_hashed",
        numFeatures=N_HASH_BUCKETS,
        categoricalCols=CAT_COLS,
    )

    # Final feature vector: hashed cats + pre-scaled numerics
    feat_assembler = VectorAssembler(
        inputCols=["cat_hashed"] + NUMERIC_FEATURE_COLS,
        outputCol="features",
        handleInvalid="skip",
    )

    # Logistic Regression
    lr = LogisticRegression(
        featuresCol="features",
        labelCol=LABEL_COL,
        regParam=reg_param,
        elasticNetParam=elastic_net,
        maxIter=max_iter,
        standardization=False,
        weightCol="class_weight",
    )

    return Pipeline(stages=[hasher, feat_assembler, lr])

print(f"✅ Pipeline builder ready (FeatureHasher with {N_HASH_BUCKETS} buckets)")

## 4 · Class imbalance handling

Positive rows get weight = neg/pos, negative rows get weight = 1.0.  
Matching the approach documented in `02_gold_features`.

In [0]:
import datetime

VAL_FRAC = 0.2

# Chronological train/val split (last 20% of train period → validation)
date_range = train_df.agg(F.min(TS_COL).alias("mn"), F.max(TS_COL).alias("mx")).first()
min_dt, max_dt = date_range["mn"], date_range["mx"]
total_sec = (max_dt - min_dt).total_seconds()
val_cutoff = min_dt + datetime.timedelta(seconds=total_sec * (1 - VAL_FRAC))
print(f"Validation cutoff: {val_cutoff}")

train_fit_df = train_df.filter(F.col(TS_COL) <  F.lit(val_cutoff))
train_val_df = train_df.filter(F.col(TS_COL) >= F.lit(val_cutoff))
print(f"Train (fit) : {train_fit_df.count():,} rows")
print(f"Validation  : {train_val_df.count():,} rows")

# Class weights for the GRID loop — computed from train_fit only
n_fit       = train_fit_df.count()
n_pos_fit   = train_fit_df.filter(F.col(LABEL_COL) == 1).count()
n_neg_fit   = n_fit - n_pos_fit
imb_fit     = n_neg_fit / builtins.max(n_pos_fit, 1)

train_fit_w = train_fit_df.withColumn(
    "class_weight",
    F.when(F.col(LABEL_COL) == 1, F.lit(imb_fit)).otherwise(F.lit(1.0))
)
train_val_w = train_val_df.withColumn("class_weight", F.lit(1.0))
test_w      = test_df.withColumn("class_weight", F.lit(1.0))

# Class weights for the FINAL refit — recomputed on full train
n_train  = train_df.count()
n_pos    = train_df.filter(F.col(LABEL_COL) == 1).count()
n_neg    = n_train - n_pos
imb_full = n_neg / builtins.max(n_pos, 1)

train_full_w = train_df.withColumn(
    "class_weight",
    F.when(F.col(LABEL_COL) == 1, F.lit(imb_full)).otherwise(F.lit(1.0))
)

train_fit_w.cache(); train_val_w.cache(); train_full_w.cache(); test_w.cache()
print(f"\nFit imbalance ratio  : {imb_fit:.4f}")
print(f"Full imbalance ratio : {imb_full:.4f}")
print("Cached ✅")

## 5 · Hyperparameter grid search (with MLflow)

Manual loop — each configuration gets its own MLflow child run.  
Primary selection metric: **AUPRC**.

In [0]:
PARAM_GRID = [
    (0.0,   0.0),    # no regularisation
    (0.01,  0.0),    # light L2
    (0.1,   0.0),    # moderate L2
    (0.01,  0.5),    # elastic net
    (0.01,  1.0),    # L1 (Lasso)
    (0.1,   0.5),    # moderate elastic net
    (0.1,   1.0),    # moderate L1
]

MAX_ITER = 200

In [0]:
mlflow.set_experiment(EXPERIMENT)

best = {"auprc": -1, "run_id": None, "params": None}

with mlflow.start_run(run_name=f"{MODEL_NAME}_{LABEL_COL}") as parent_run:
    mlflow.log_param("model_type", MODEL_NAME)
    mlflow.log_param("label", LABEL_COL)
    mlflow.log_param("n_numeric_features", len(NUMERIC_FEATURE_COLS))
    mlflow.log_param("cat_cols", str(CAT_COLS))
    mlflow.log_param("imbalance_ratio_fit",  builtins.round(imb_fit, 4))
    mlflow.log_param("imbalance_ratio_full", builtins.round(imb_full, 4))
    mlflow.log_param("val_frac", VAL_FRAC)
    mlflow.log_param("val_cutoff", str(val_cutoff))
    mlflow.log_param("selection_metric", "AUPRC_on_validation")

    for reg_param, elastic_net in PARAM_GRID:
        run_name = f"reg={reg_param}_enet={elastic_net}"

        with mlflow.start_run(run_name=run_name, nested=True) as child_run:
            mlflow.log_param("regParam", reg_param)
            mlflow.log_param("elasticNetParam", elastic_net)
            mlflow.log_param("maxIter", MAX_ITER)

            pipe  = make_lr_pipeline(reg_param=reg_param, elastic_net=elastic_net, max_iter=MAX_ITER)
            model = pipe.fit(train_fit_w)

            # Evaluate on VAL (not test)
            val_preds = extract_prob_positive(model.transform(train_val_w))
            local     = val_preds.select(F.col(LABEL_COL).cast("int"), "prob_pos").toPandas()
            y_val, p_val = local[LABEL_COL].values, local["prob_pos"].values

            val_metrics = evaluate_binary(y_val, p_val, threshold=0.5,
                                          title=f"{MODEL_NAME} | {run_name} | VAL")
            log_evaluation_to_mlflow(val_metrics, y_val, p_val, threshold=0.5, prefix="val")

            if val_metrics["AUPRC"] > best["auprc"]:
                best = {"auprc": val_metrics["AUPRC"],
                        "run_id": child_run.info.run_id,
                        "params": (reg_param, elastic_net)}

    mlflow.log_param("best_regParam",   best["params"][0])
    mlflow.log_param("best_elasticNet", best["params"][1])
    mlflow.log_metric("best_val_AUPRC", best["auprc"])

print(f"\n🏆 Best on VAL: regParam={best['params'][0]}, elasticNet={best['params'][1]} "
      f"| val AUPRC={best['auprc']:.4f}")

## 6 · Best model — full evaluation on test set

In [0]:
while mlflow.active_run() is not None:
    mlflow.end_run()

In [0]:
best_reg, best_enet = best["params"]

with mlflow.start_run(run_id=parent_run.info.run_id):
    final_pipe  = make_lr_pipeline(reg_param=best_reg, elastic_net=best_enet, max_iter=MAX_ITER)
    final_model = final_pipe.fit(train_full_w)

    test_preds = extract_prob_positive(final_model.transform(test_w))
    local      = test_preds.select(F.col(LABEL_COL).cast("int"), "prob_pos").toPandas()
    y_true = local[LABEL_COL].values
    y_prob = local["prob_pos"].values

    final_metrics = evaluate_binary(y_true, y_prob, threshold=0.5,
                                    title=f"FINAL TEST — {MODEL_NAME} ({LABEL_COL})")
    log_evaluation_to_mlflow(final_metrics, y_true, y_prob, threshold=0.5, prefix="test")

    with mlflow.start_run(run_name="final_refit", nested=True):
        mlflow.spark.log_model(final_model, artifact_path="model")

print(f"\nTest AUPRC: {final_metrics['AUPRC']:.4f}")

## 7 · Threshold sweep

In [0]:
sweep_results = threshold_sweep(y_true, y_prob)
display(spark.createDataFrame(sweep_results).orderBy(F.desc("f1")))

In [0]:
prec, rec, thrs = precision_recall_curve(y_true, y_prob)
f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-8)
best_idx = min(np.argmax(f1), len(thrs) - 1)

In [0]:
optimal_threshold = float(thrs[best_idx])
print(f"Optimal threshold (max F1): {optimal_threshold}")

optimal_metrics = evaluate_binary(y_true, y_prob, threshold=optimal_threshold, title=f"TEST @ threshold={optimal_threshold} — {MODEL_NAME}")

In [0]:
# best_thr_row = builtins.max(sweep_results, key=lambda r: r["f1"])
# optimal_threshold = best_thr_row["threshold"]
# print(f"Optimal threshold (max F1): {optimal_threshold}")

# optimal_metrics = evaluate_binary(y_true, y_prob, threshold=optimal_threshold, title=f"TEST @ threshold={optimal_threshold} — {MODEL_NAME}")

## 8 · Confusion matrix & curves

In [0]:
display(plot_confusion_matrix(y_true, (y_prob >= 0.5).astype(int), title=f"{MODEL_NAME} — CM @ 0.5"))
display(plot_confusion_matrix(y_true, (y_prob >= optimal_threshold).astype(int), title=f"{MODEL_NAME} — CM @ {optimal_threshold}"))

In [0]:
plot_roc_curve(y_true, y_prob, title=f"{MODEL_NAME} — ROC")

In [0]:
display(plot_pr_curve(y_true, y_prob, title=f"{MODEL_NAME} — Precision-Recall"))

## 9 · Feature importance (individual + grouped)

For LR, importance = |coefficient|.  
Grouping and plotting functions come from `00_Evaluation`.

In [0]:
from pyspark.ml.classification import LogisticRegressionModel
best_model = final_model
lr_model = None
for stage in best_model.stages:
    if isinstance(stage, LogisticRegressionModel):
        lr_model = stage
        break

assert lr_model is not None, "Could not find LR stage in pipeline"

coeffs = lr_model.coefficients.toArray()
print(f"Intercept: {lr_model.intercept:.4f}")
print(f"Number of coefficients: {len(coeffs)}")

# Recover feature names from metadata
pred_one = best_model.transform(test_w.limit(1))
feat_meta = pred_one.schema["features"].metadata

feat_names = []
if "ml_attr" in feat_meta and "attrs" in feat_meta["ml_attr"]:
    for attr_type in feat_meta["ml_attr"]["attrs"]:
        for attr in feat_meta["ml_attr"]["attrs"][attr_type]:
            feat_names.append((attr["idx"], attr["name"]))
    feat_names.sort(key=lambda x: x[0])
    feat_names = [n for _, n in feat_names]
else:
    feat_names = [f"f_{i}" for i in range(len(coeffs))]

print(f"Feature names recovered: {len(feat_names)}")

In [0]:
fig_top = plot_top_features(feat_names, coeffs, top_n=20, title=f"{MODEL_NAME} — Top 20 by |Coefficient|", xlabel="|Coefficient|")
display(fig_top)

# Table view
coeff_rows = list(zip(feat_names, [float(c) for c in coeffs], [float(np.abs(c)) for c in coeffs], [classify_feature(n) for n in feat_names]))
coeff_df = spark.createDataFrame(coeff_rows, ["feature", "coefficient", "abs_coeff", "group"])
display(coeff_df.orderBy(F.desc("abs_coeff")).limit(20))

In [0]:
grp = grouped_importance(feat_names, coeffs)
fig_grp = plot_grouped_importance(grp, title=f"{MODEL_NAME} — Feature Group Importance", xlabel="Sum |Coefficient|")
display(fig_grp)

grp_rows = [
    (g, d["n_dims"], float(d["sum"]), float(d["mean"]))
    for g, d in grp.items()
]
display(spark.createDataFrame(grp_rows, ["group", "n_dims", "sum_abs_coeff", "mean_abs_coeff"]).orderBy(F.desc("sum_abs_coeff")))

In [0]:
with mlflow.start_run(run_id=best["run_id"]):
    mlflow.log_figure(fig_top, "feature_importance_top20.png")
    mlflow.log_figure(fig_grp, "feature_importance_grouped.png")
    mlflow.log_table(
    pd.DataFrame([dict(group=g, **d) for g, d in grp.items()]),
    artifact_file="grouped_feature_importance.json",)

plt.close(fig_top)
plt.close(fig_grp)
print("✅ Feature importance logged to MLflow")